In [0]:
dbutils.widgets.removeAll()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import functions as F

In [0]:
dbutils.widgets.text("catalogo", "catalog_ecobici")
dbutils.widgets.text("esquema_source", "silver")
dbutils.widgets.text("esquema_sink", "golden")

In [0]:
catalogo = dbutils.widgets.get("catalogo")
esquema_source = dbutils.widgets.get("esquema_source")
esquema_sink = dbutils.widgets.get("esquema_sink")

In [0]:
df_ecobici = spark.read.table(f"{catalogo}.{esquema_source}.ecobici")
df_weather = spark.read.table(f"{catalogo}.{esquema_source}.weather")

In [0]:
df_ecobici = df_ecobici.withColumn("date", to_date(col("date")))
df_weather = df_weather.withColumn("date", to_date(col("date")))

df_join = df_ecobici.join(df_weather, "date")

## 🔹 GOLDEN LLUVIA

In [0]:
df_lluvia = df_join.groupBy("lluvia") \
    .agg(
        sum("total_viajes").alias("total_viajes"),
        sum("viajes_hombres").alias("viajes_hombres"),
        sum("viajes_mujeres").alias("viajes_mujeres"),
        sum("viajes_otros").alias("viajes_otros")
    )

In [0]:
df_lluvia.write \
    .mode("overwrite") \
    .saveAsTable(f"{catalogo}.{esquema_sink}.ecobici_vs_lluvia")

## 🔹 GOLDEN TEMPERATURA

In [0]:
df_temp = df_join.groupBy("temp_avg") \
    .agg(
        sum("total_viajes").alias("total_viajes"),
        sum("viajes_hombres").alias("viajes_hombres"),
        sum("viajes_mujeres").alias("viajes_mujeres"),
        sum("viajes_otros").alias("viajes_otros")
    ) \
    .withColumnRenamed("temp_avg", "temperatura")

In [0]:
df_temp.write \
    .mode("overwrite") \
    .saveAsTable(f"{catalogo}.{esquema_sink}.ecobici_vs_temperatura")

## VALIDACIONES — GOLDEN

In [0]:
%sql
SELECT COUNT(*) FROM catalog_ecobici.golden.ecobici_vs_lluvia;


In [0]:
%sql
SELECT COUNT(*) FROM catalog_ecobici.golden.ecobici_vs_temperatura;

In [0]:
%sql
SELECT 
    (SELECT SUM(total_viajes) FROM catalog_ecobici.silver.ecobici) AS total_silver,
    (SELECT SUM(total_viajes) FROM catalog_ecobici.golden.ecobici_vs_lluvia) AS total_golden;

In [0]:
%sql
SELECT lluvia, total_viajes
FROM catalog_ecobici.golden.ecobici_vs_lluvia
ORDER BY lluvia;

In [0]:
%sql
SELECT temperatura, total_viajes
FROM catalog_ecobici.golden.ecobici_vs_temperatura
ORDER BY temperatura;

In [0]:
%sql
SELECT COUNT(*) AS registros_join
FROM catalog_ecobici.silver.ecobici e
JOIN catalog_ecobici.silver.weather w
ON e.date = w.date;

In [0]:
%sql
SELECT lluvia, COUNT(*) 
FROM catalog_ecobici.golden.ecobici_vs_lluvia
GROUP BY lluvia
HAVING COUNT(*) > 1;

In [0]:
%sql
SELECT 
    COUNT(DISTINCT date) AS dias_silver,
    (SELECT COUNT(*) FROM catalog_ecobici.golden.ecobici_vs_lluvia) AS buckets_lluvia
FROM catalog_ecobici.silver.ecobici;

In [0]:
%sql
SELECT COUNT(*) FROM catalog_ecobici.silver.ecobici;


In [0]:
%sql
SELECT COUNT(*) FROM catalog_ecobici.silver.weather;

In [0]:
%sql

SELECT COUNT(*) FROM catalog_ecobici.golden.ecobici_vs_lluvia;

In [0]:
%sql
SHOW TABLES IN catalog_ecobici.golden;